In [ ]:
# Import essential libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

# Set visualization styles
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
np.random.seed(42)

## 1. Import Required Libraries

We'll import essential libraries for data manipulation, modeling, and visualization:

# Housing Price Prediction with Linear Regression

This notebook demonstrates a complete machine learning workflow for predicting house prices using linear regression. We will cover data exploration, preprocessing, model training, evaluation, and interpretation.

**Dataset**: Kaggle Housing Price Prediction Dataset
**Objective**: Build a predictive model to estimate house prices based on relevant features
**Model**: Linear Regression (Scikit-Learn)

## 2. Load and Explore the Housing Dataset

Let's load the dataset and examine its basic structure:

In [ ]:
# Load the dataset
# Note: Download the data from Kaggle and place it in the data/ folder
# or upload it here if using Kaggle notebook

try:
    # Try loading from the data folder
    df = pd.read_csv('../data/train.csv')
    print("Dataset loaded successfully!")
except FileNotFoundError:
    print("Please download the dataset from Kaggle and place it in the data/ folder")
    print("Expected path: ../data/train.csv")
    df = None

if df is not None:
    # Display basic information
    print("\n" + "="*50)
    print("DATASET SHAPE AND INFO")
    print("="*50)
    print(f"Dataset shape: {df.shape}")
    print(f"\nData Types:\n{df.dtypes}")
    
    print("\n" + "="*50)
    print("FIRST FEW ROWS")
    print("="*50)
    print(df.head())
    
    print("\n" + "="*50)
    print("STATISTICAL SUMMARY")
    print("="*50)
    print(df.describe())

## 3. Data Cleaning and Preprocessing

Handle missing values, duplicates, and ensure data quality:

In [ ]:
if df is not None:
    print("="*50)
    print("CHECKING FOR MISSING VALUES")
    print("="*50)
    missing_values = df.isnull().sum()
    if missing_values.sum() > 0:
        print(f"Missing values found:\n{missing_values[missing_values > 0]}")
        print(f"\nPercentage of missing values:\n{(missing_values[missing_values > 0] / len(df) * 100).round(2)}")
        
        # Handle missing values
        # Fill numerical columns with median
        numerical_cols = df.select_dtypes(include=[np.number]).columns
        for col in numerical_cols:
            if df[col].isnull().sum() > 0:
                df[col].fillna(df[col].median(), inplace=True)
        
        # Fill categorical columns with mode
        categorical_cols = df.select_dtypes(include=['object']).columns
        for col in categorical_cols:
            if df[col].isnull().sum() > 0:
                df[col].fillna(df[col].mode()[0], inplace=True)
        
        print("\nMissing values handled successfully!")
    else:
        print("No missing values found!")
    
    print("\n" + "="*50)
    print("CHECKING FOR DUPLICATES")
    print("="*50)
    duplicates = df.duplicated().sum()
    print(f"Number of duplicate rows: {duplicates}")
    if duplicates > 0:
        df.drop_duplicates(inplace=True)
        print(f"Duplicates removed. New shape: {df.shape}")
    
    print("\nData cleaning completed!")

## 4. Exploratory Data Analysis (EDA) and Visualization

Create visualizations to understand feature distributions and relationships:

In [ ]:
if df is not None:
    # Identify numerical and categorical columns
    numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
    
    print(f"Numerical columns: {numerical_cols}")
    print(f"Categorical columns: {categorical_cols}")
    
    # Identify target variable (usually the last or 'Price' column)
    # Adjust this based on your actual dataset
    if 'Price' in numerical_cols:
        target = 'Price'
    elif 'SalePrice' in numerical_cols:
        target = 'SalePrice'
    else:
        target = numerical_cols[-1]
    
    print(f"\nTarget variable: {target}")
    
    # Distribution of target variable
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Histogram
    axes[0].hist(df[target], bins=30, color='skyblue', edgecolor='black')
    axes[0].set_title(f'Distribution of {target}', fontsize=12, fontweight='bold')
    axes[0].set_xlabel(target)
    axes[0].set_ylabel('Frequency')
    
    # Box plot
    axes[1].boxplot(df[target], vert=True)
    axes[1].set_title(f'Box Plot of {target}', fontsize=12, fontweight='bold')
    axes[1].set_ylabel(target)
    
    plt.tight_layout()
    plt.show()
    
    # Distribution of other numerical features
    n_cols = len([col for col in numerical_cols if col != target])
    if n_cols > 0:
        n_rows = (n_cols + 3) // 4
        fig, axes = plt.subplots(n_rows, 4, figsize=(16, n_rows * 4))
        axes = axes.flatten() if n_cols > 1 else [axes]
        
        for idx, col in enumerate([col for col in numerical_cols if col != target][:min(n_cols, len(axes))]):
            axes[idx].hist(df[col], bins=20, color='lightcoral', edgecolor='black')
            axes[idx].set_title(f'Distribution of {col}', fontsize=10, fontweight='bold')
            axes[idx].set_xlabel(col)
            axes[idx].set_ylabel('Frequency')
        
        # Hide extra subplots
        for idx in range(n_cols, len(axes)):
            axes[idx].set_visible(False)
        
        plt.tight_layout()
        plt.show()

### Correlation Analysis

Analyze correlations between numerical features and the target variable:

In [ ]:
if df is not None:
    # Correlation matrix
    correlation_matrix = df[numerical_cols].corr()
    
    # Correlation with target variable
    target_correlation = correlation_matrix[target].sort_values(ascending=False)
    print(f"\nCorrelation with {target}:")
    print(target_correlation)
    
    # Heatmap of correlation matrix
    plt.figure(figsize=(12, 10))
    sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0)
    plt.title('Correlation Matrix of Numerical Features', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    # Scatter plots for top correlated features
    top_features = target_correlation.drop(target).head(5).index.tolist()
    print(f"\nTop 5 features correlated with {target}: {top_features}")
    
    fig, axes = plt.subplots(1, min(5, len(top_features)), figsize=(16, 4))
    if len(top_features) == 1:
        axes = [axes]
    
    for idx, feature in enumerate(top_features[:min(5, len(axes))]):
        axes[idx].scatter(df[feature], df[target], alpha=0.6, color='green')
        axes[idx].set_xlabel(feature)
        axes[idx].set_ylabel(target)
        axes[idx].set_title(f'{feature} vs {target}')
        # Add correlation coefficient
        corr_coef = df[feature].corr(df[target])
        axes[idx].text(0.05, 0.95, f'Corr: {corr_coef:.3f}', 
                      transform=axes[idx].transAxes, verticalalignment='top',
                      bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    plt.tight_layout()
    plt.show()

## 5. Feature Selection and Engineering

Select relevant features and handle categorical variables:

In [ ]:
if df is not None:
    # Select features based on correlation with target
    # Keep features with correlation > 0.1 or < -0.1 (adjust threshold as needed)
    correlation_threshold = 0.1
    relevant_features = target_correlation[
        (target_correlation.abs() > correlation_threshold) & (target_correlation.index != target)
    ].index.tolist()
    
    print(f"Features with correlation > {correlation_threshold} (absolute): {len(relevant_features)}")
    print(relevant_features)
    
    # Handle categorical variables
    if categorical_cols:
        print(f"\nCategorical columns found: {categorical_cols}")
        # One-hot encoding for categorical variables
        df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)
        print(f"After encoding, dataset shape: {df_encoded.shape}")
    else:
        df_encoded = df.copy()
        print("No categorical columns to encode")
    
    # Prepare features (X) and target (y)
    # Use all numerical columns except target as features
    X_cols = [col for col in df_encoded.columns if col != target and df_encoded[col].dtype in [np.int64, np.float64]]
    X = df_encoded[X_cols]
    y = df_encoded[target]
    
    print(f"\nFeatures shape: {X.shape}")
    print(f"Target shape: {y.shape}")
    print(f"\nFeatures used: {X_cols[:10]}...")  # Display first 10 features
    
    # Feature scaling (normalize)
    from sklearn.preprocessing import StandardScaler
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    X_scaled = pd.DataFrame(X_scaled, columns=X_cols)
    
    print(f"\nFeatures normalized successfully!")
    print(f"Feature statistics after scaling:\n{X_scaled.describe()}")

## 6. Split Data into Training and Testing Sets

Divide the dataset into training (80%) and testing (20%) subsets:

In [ ]:
if df is not None:
    # Split data into training and testing sets
    X_train, X_test, y_train, y_test = train_test_split(
        X_scaled, y, test_size=0.2, random_state=42
    )
    
    print("="*50)
    print("DATA SPLIT SUMMARY")
    print("="*50)
    print(f"Total samples: {len(X_scaled)}")
    print(f"Training samples: {len(X_train)} ({len(X_train)/len(X_scaled)*100:.1f}%)")
    print(f"Testing samples: {len(X_test)} ({len(X_test)/len(X_scaled)*100:.1f}%)")
    print(f"\nFeatures shape: {X_train.shape}")
    print(f"Target shape: {y_train.shape}")
    
    # Verify data integrity
    print(f"\nTraining target statistics:")
    print(y_train.describe())
    print(f"\nTesting target statistics:")
    print(y_test.describe())

## 7. Train Linear Regression Model

Instantiate and fit a LinearRegression model using scikit-learn:

In [ ]:
if df is not None:
    # Create and train the Linear Regression model
    model = LinearRegression()
    
    print("Training Linear Regression Model...")
    model.fit(X_train, y_train)
    print("Model training completed!")
    
    # Model information
    print(f"\nModel coefficients shape: {model.coef_.shape}")
    print(f"Model intercept: {model.intercept_:.4f}")
    
    # Display training metrics
    y_train_pred = model.predict(X_train)
    train_r2 = r2_score(y_train, y_train_pred)
    print(f"\nTraining R² Score: {train_r2:.4f}")

## 8. Make Predictions on Test Data

Use the trained model to generate predictions on the test dataset:

In [ ]:
if df is not None:
    # Make predictions on test data
    y_pred = model.predict(X_test)
    
    # Create a comparison dataframe
    comparison_df = pd.DataFrame({
        'Actual': y_test.values,
        'Predicted': y_pred,
        'Difference': y_test.values - y_pred,
        'Absolute_Error': np.abs(y_test.values - y_pred),
        'Percentage_Error': np.abs((y_test.values - y_pred) / y_test.values * 100)
    })
    
    print("="*70)
    print("PREDICTIONS VS ACTUAL VALUES (First 10 samples)")
    print("="*70)
    print(comparison_df.head(10))
    
    print("\n" + "="*70)
    print("PREDICTION ERROR STATISTICS")
    print("="*70)
    print(comparison_df[['Difference', 'Absolute_Error', 'Percentage_Error']].describe())

## 9. Evaluate Model Performance

Calculate comprehensive evaluation metrics:

In [ ]:
if df is not None:
    # Calculate evaluation metrics
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    # Calculate additional metrics
    rss = np.sum((y_test - y_pred) ** 2)  # Residual Sum of Squares
    tss = np.sum((y_test - y_test.mean()) ** 2)  # Total Sum of Squares
    adjusted_r2 = 1 - ((1 - r2) * (len(y_test) - 1) / (len(y_test) - X_test.shape[1] - 1))
    
    print("="*70)
    print("MODEL EVALUATION METRICS")
    print("="*70)
    print(f"Mean Absolute Error (MAE):       {mae:>15,.4f}")
    print(f"Mean Squared Error (MSE):        {mse:>15,.4f}")
    print(f"Root Mean Squared Error (RMSE):  {rmse:>15,.4f}")
    print(f"R² Score:                        {r2:>15,.4f}")
    print(f"Adjusted R² Score:               {adjusted_r2:>15,.4f}")
    print(f"Residual Sum of Squares (RSS):   {rss:>15,.4f}")
    
    # Calculate MAPE (Mean Absolute Percentage Error)
    mape = np.mean(comparison_df['Percentage_Error'])
    print(f"Mean Absolute % Error (MAPE):    {mape:>15,.4f}%")
    
    print("\n" + "="*70)
    print("MODEL INTERPRETATION")
    print("="*70)
    print(f"The model explains {r2*100:.2f}% of the variance in house prices.")
    print(f"On average, predictions are off by ${mae:,.2f} from actual values.")
    print(f"RMSE of ${rmse:,.2f} indicates typical prediction deviation.")

## 10. Visualize Predictions vs Actual Values

Create comprehensive visualizations for model predictions:

In [ ]:
if df is not None:
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    
    # 1. Actual vs Predicted Scatter Plot
    ax1 = axes[0, 0]
    ax1.scatter(y_test, y_pred, alpha=0.6, color='blue', edgecolors='black')
    # Add perfect prediction line
    min_val = min(y_test.min(), y_pred.min())
    max_val = max(y_test.max(), y_pred.max())
    ax1.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Perfect Prediction')
    ax1.set_xlabel('Actual Values', fontsize=11, fontweight='bold')
    ax1.set_ylabel('Predicted Values', fontsize=11, fontweight='bold')
    ax1.set_title('Actual vs Predicted Values', fontsize=12, fontweight='bold')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # 2. Residuals Plot
    residuals = y_test.values - y_pred
    ax2 = axes[0, 1]
    ax2.scatter(y_pred, residuals, alpha=0.6, color='green', edgecolors='black')
    ax2.axhline(y=0, color='r', linestyle='--', lw=2)
    ax2.set_xlabel('Predicted Values', fontsize=11, fontweight='bold')
    ax2.set_ylabel('Residuals', fontsize=11, fontweight='bold')
    ax2.set_title('Residual Plot', fontsize=12, fontweight='bold')
    ax2.grid(True, alpha=0.3)
    
    # 3. Distribution of Residuals
    ax3 = axes[1, 0]
    ax3.hist(residuals, bins=30, color='orange', edgecolor='black', alpha=0.7)
    ax3.set_xlabel('Residuals', fontsize=11, fontweight='bold')
    ax3.set_ylabel('Frequency', fontsize=11, fontweight='bold')
    ax3.set_title('Distribution of Residuals (Should be Normal)', fontsize=12, fontweight='bold')
    ax3.axvline(x=0, color='r', linestyle='--', lw=2)
    ax3.grid(True, alpha=0.3, axis='y')
    
    # 4. Error Distribution
    ax4 = axes[1, 1]
    ax4.scatter(range(len(comparison_df)), comparison_df['Absolute_Error'], 
               alpha=0.6, color='purple', edgecolors='black')
    ax4.axhline(y=mae, color='r', linestyle='--', lw=2, label=f'Mean Error = ${mae:,.2f}')
    ax4.set_xlabel('Sample Index', fontsize=11, fontweight='bold')
    ax4.set_ylabel('Absolute Error ($)', fontsize=11, fontweight='bold')
    ax4.set_title('Absolute Prediction Error by Sample', fontsize=12, fontweight='bold')
    ax4.legend()
    ax4.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Q-Q Plot for residual normality check
    from scipy import stats
    fig, ax = plt.subplots(1, 1, figsize=(10, 6))
    stats.probplot(residuals, dist="norm", plot=ax)
    ax.set_title('Q-Q Plot: Testing for Normal Distribution of Residuals', 
                fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()

## 11. Interpret Model Coefficients

Analyze feature coefficients to understand their impact on house prices:

In [ ]:
if df is not None:
    # Create coefficients dataframe
    coefficients_df = pd.DataFrame({
        'Feature': X_cols,
        'Coefficient': model.coef_,
        'Abs_Coefficient': np.abs(model.coef_)
    }).sort_values('Abs_Coefficient', ascending=False)
    
    print("="*70)
    print("MODEL COEFFICIENTS INTERPRETATION")
    print("="*70)
    print(f"Model Intercept: ${model.intercept_:,.2f}\n")
    
    print("Top 15 Most Important Features (by absolute coefficient):")
    print(coefficients_df.head(15).to_string(index=False))
    
    print("\n" + "="*70)
    print("COEFFICIENT INTERPRETATION")
    print("="*70)
    print("Positive coefficient: Feature increase → Price increase")
    print("Negative coefficient: Feature increase → Price decrease")
    print("Larger absolute coefficient: Stronger impact on price\n")
    
    # Top positive coefficients (features that increase price)
    top_positive = coefficients_df[coefficients_df['Coefficient'] > 0].head(5)
    print("Top 5 Features that INCREASE house price:")
    for idx, row in top_positive.iterrows():
        print(f"  • {row['Feature']}: +${row['Coefficient']:,.2f} per unit increase")
    
    # Top negative coefficients (features that decrease price)
    top_negative = coefficients_df[coefficients_df['Coefficient'] < 0].head(5)
    print("\nTop 5 Features that DECREASE house price:")
    for idx, row in top_negative.iterrows():
        print(f"  • {row['Feature']}: ${row['Coefficient']:,.2f} per unit increase")
    
    # Visualization of coefficients
    fig, ax = plt.subplots(figsize=(12, 8))
    
    top_n = 15
    coef_data = coefficients_df.head(top_n)
    colors = ['green' if x > 0 else 'red' for x in coef_data['Coefficient']]
    
    ax.barh(range(len(coef_data)), coef_data['Coefficient'], color=colors, alpha=0.7, edgecolor='black')
    ax.set_yticks(range(len(coef_data)))
    ax.set_yticklabels(coef_data['Feature'])
    ax.set_xlabel('Coefficient Value', fontsize=11, fontweight='bold')
    ax.set_title(f'Top {top_n} Feature Coefficients (Impact on House Price)', 
                fontsize=12, fontweight='bold')
    ax.axvline(x=0, color='black', linestyle='-', lw=1)
    ax.grid(True, alpha=0.3, axis='x')
    
    plt.tight_layout()
    plt.show()

## Summary and Key Insights

### Model Performance Summary
- **R² Score**: Indicates how well the model explains the variance in house prices
- **RMSE**: Root Mean Squared Error shows the average deviation of predictions
- **MAE**: Mean Absolute Error provides interpretable error in dollar amounts

### Key Findings
1. The linear regression model captures the relationship between features and house prices
2. Feature coefficients indicate which factors have the strongest impact on pricing
3. Residual analysis helps identify if the model assumptions are met
4. The Q-Q plot shows whether residuals are normally distributed

### Next Steps for Improvement
1. **Feature Engineering**: Create polynomial features or interaction terms
2. **Model Selection**: Try Ridge or Lasso regression for regularization
3. **Additional Models**: Compare with other algorithms (Random Forest, Gradient Boosting)
4. **Hyperparameter Tuning**: Optimize model parameters using cross-validation
5. **Feature Importance**: Use techniques like SHAP for deeper feature analysis
6. **Outlier Treatment**: Identify and handle outliers that affect model performance
7. **Cross-Validation**: Implement k-fold cross-validation for robust evaluation

### Learning Objectives Achieved ✓
- ✅ Understanding of linear regression concepts
- ✅ Complete implementation of a predictive model
- ✅ Data exploration and preprocessing techniques
- ✅ Comprehensive model evaluation and interpretation
- ✅ Professional data visualization and analysis